# Reliable RAG

<img src="./images/reliable_rag.svg" alt="Reliable-RAG" width="300">

## Overview

Standard RAG retrieves documents and feeds them straight into the LLM — even if the documents are irrelevant. **Reliable RAG** adds three safety checks:

| Step | What it checks | Why it matters |
|---|---|---|
| **Relevance grading** | Is each retrieved document actually related to the question? | Filters out noise before generation |
| **Hallucination check** | Is the generated answer grounded in the retrieved documents? | Catches fabricated facts |
| **Source highlighting** | Which exact segments from the docs were used to answer? | Provides transparency and traceability |

## Models Used

- **LLM**: `gemma3:4b` via Ollama (local)
- **Embeddings**: `mxbai-embed-large:335m` via Ollama (local)

---
## Step 0: Import Packages

In [1]:
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.document_loaders import WebBaseLoader
from langchain_community.vectorstores import Chroma
from langchain_ollama.embeddings import OllamaEmbeddings
from langchain_ollama import ChatOllama
from langchain_core.prompts import ChatPromptTemplate, PromptTemplate
from langchain_core.output_parsers import StrOutputParser
from IPython.display import display, HTML

USER_AGENT environment variable not set, consider setting it to identify your requests.


---
## Step 1: Set Up LLM and Embedding Model

In [2]:
embedding_model = OllamaEmbeddings(model="mxbai-embed-large:335m")
llm = ChatOllama(model="gemma3:4b", temperature=0)

print("LLM and embedding model ready")

LLM and embedding model ready


---
## Step 2: Load Web Pages and Create the Vector Store

We load 5 articles from DeepLearning.AI about agentic design patterns, split them into chunks, and store them in ChromaDB.

In [3]:
urls = [
    "https://www.deeplearning.ai/the-batch/how-agents-can-improve-llm-performance/?ref=dl-staging-website.ghost.io",
    "https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-2-reflection/?ref=dl-staging-website.ghost.io",
    "https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-3-tool-use/?ref=dl-staging-website.ghost.io",
    "https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-4-planning/?ref=dl-staging-website.ghost.io",
    "https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-5-multi-agent-collaboration/?ref=dl-staging-website.ghost.io",
]

docs_list = []
for url in urls:
    loaded = WebBaseLoader(url).load()
    docs_list.extend(loaded)

print(f"Loaded {len(docs_list)} web pages")

text_splitter = RecursiveCharacterTextSplitter.from_tiktoken_encoder(
    chunk_size=1000, chunk_overlap=100
)
doc_splits = text_splitter.split_documents(docs_list)
print(f"Split into {len(doc_splits)} chunks")

vectorstore = Chroma.from_documents(doc_splits, embedding_model)
print("Vector store created")

Loaded 5 web pages
Split into 8 chunks
Vector store created


---
## Step 3: Retrieve Documents

We create a retriever that returns the top-4 most similar chunks to the question.

In [4]:
question = "what are the different kind of agentic design patterns?"
print(f"Question: {question}\n")

retriever = vectorstore.as_retriever(search_type="similarity", search_kwargs={"k": 4})
docs = retriever.invoke(question)

print(f"Retrieved {len(docs)} documents:\n")
for i, doc in enumerate(docs):
    display(HTML(f"<span style='background-color: lightblue;'> <b>Context {i}:</b></span>"))
    display(HTML(f"<b>Page Content:</b> {doc.page_content[:200]}..."))
    display(HTML(f"<b>Source:</b> {doc.metadata.get('source', 'N/A')}"))
    print("=" * 80)

Question: what are the different kind of agentic design patterns?

Retrieved 4 documents:



---
## Step 4: Grade Document Relevance

Not every retrieved document is actually useful. We ask the LLM to grade each document: **"yes"** (relevant) or **"no"** (not relevant), then keep only the relevant ones.

This uses structured output with a JSON schema (no Pydantic class needed).

In [5]:
grade_schema = {
    "title": "GradeDocuments",
    "description": "Binary score for relevance check on retrieved documents.",
    "type": "object",
    "properties": {
        "binary_score": {
            "type": "string",
            "description": "Documents are relevant to the question, 'yes' or 'no'"
        }
    },
    "required": ["binary_score"]
}

grade_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a grader assessing relevance of a retrieved document to a user question. "
     "If the document contains keyword(s) or semantic meaning related to the user question, grade it as relevant. "
     "It does not need to be a stringent test. The goal is to filter out erroneous retrievals. "
     "Give a binary score 'yes' or 'no' score to indicate whether the document is relevant to the question."),
    ("human", "Retrieved document: \n\n {document} \n\n User question: {question}"),
])

retrieval_grader = grade_prompt | llm.with_structured_output(grade_schema)

# Grade each retrieved document and keep only relevant ones
docs_to_use = []
for i, doc in enumerate(docs):
    result = retrieval_grader.invoke({"question": question, "document": doc.page_content})
    score = result["binary_score"].lower().strip()
    display(HTML(f"<span style='background-color: lightblue;'> <b>Context {i}:</b> relevance = <b>{score}</b></span>"))
    if score == "yes":
        docs_to_use.append(doc)

print(f"\nKept {len(docs_to_use)} out of {len(docs)} documents")


Kept 4 out of 4 documents


---
## Step 5: Generate the Answer from Relevant Documents

Now we feed only the **relevant** documents into the LLM to generate an answer.

In [6]:
# Format relevant docs into a single string for the prompt
formatted_docs = "\n".join(
    f"<doc{i+1}>:\nTitle: {doc.metadata.get('title', 'Untitled')}\n"
    f"Source: {doc.metadata.get('source', 'N/A')}\n"
    f"Content: {doc.page_content}\n</doc{i+1}>"
    for i, doc in enumerate(docs_to_use)
)

answer_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are an assistant for question-answering tasks. Answer the question based upon your knowledge. "
     "Use three-to-five sentences maximum and keep the answer concise."),
    ("human",
     "Retrieved documents: \n\n <docs>{documents}</docs> \n\n User question: <question>{question}</question>"),
])

answer_chain = answer_prompt | llm | StrOutputParser()
generation = answer_chain.invoke({"documents": formatted_docs, "question": question})

print(f"Question: {question}")
print(f"\nAnswer: {generation}")

Question: what are the different kind of agentic design patterns?

Answer: Based on the provided documents, the key agentic design patterns are:

1.  **Multi-Agent Collaboration:** This involves breaking down complex tasks into subtasks handled by different AI agents, allowing for more efficient and specialized execution.
2.  **Planning:** This pattern utilizes an LLM to autonomously determine the sequence of steps needed to achieve a larger goal.
3.  **Tool Use:** This pattern enables LLMs to interact with external tools – such as web search engines, code interpreters, or productivity applications – to gather information, take action, or manipulate data. 

The documents highlight that these patterns are increasingly important as LLMs evolve and become more capable of interacting with the real world.


---
## Step 6: Check for Hallucination

Even with relevant documents, the LLM might hallucinate facts not present in the context. We ask another grader: **"Is the answer grounded in the documents?"** — `yes` or `no`.

In [7]:
hallucination_schema = {
    "title": "GradeHallucinations",
    "description": "Binary score for hallucination present in generation answer.",
    "type": "object",
    "properties": {
        "binary_score": {
            "type": "string",
            "description": "Answer is grounded in the facts, 'yes' or 'no'"
        }
    },
    "required": ["binary_score"]
}

hallucination_prompt = ChatPromptTemplate.from_messages([
    ("system",
     "You are a grader assessing whether an LLM generation is grounded in / supported by a set of retrieved facts. "
     "Give a binary score 'yes' or 'no'. 'Yes' means that the answer is grounded in / supported by the set of facts."),
    ("human",
     "Set of facts: \n\n <facts>{documents}</facts> \n\n LLM generation: <generation>{generation}</generation>"),
])

hallucination_grader = hallucination_prompt | llm.with_structured_output(hallucination_schema)

result = hallucination_grader.invoke({"documents": formatted_docs, "generation": generation})
grounded = result["binary_score"].lower().strip()

if grounded == "yes":
    print(f"Hallucination check: PASSED (answer is grounded in the documents)")
else:
    print(f"Hallucination check: FAILED (answer may contain hallucinated content)")
print(f"Raw result: {result}")

Hallucination check: PASSED (answer is grounded in the documents)
Raw result: {'binary_score': 'yes'}


---
## Step 7: Highlight Source Segments

Finally, we ask the LLM to identify **which exact text segments** from the documents were used to produce the answer. This provides full traceability.

In [8]:
highlight_schema = {
    "title": "HighlightDocuments",
    "description": "Return the specific parts of documents used for answering the question.",
    "type": "object",
    "properties": {
        "id": {
            "type": "array",
            "items": {"type": "string"},
            "description": "List of doc IDs used to answer the question"
        },
        "title": {
            "type": "array",
            "items": {"type": "string"},
            "description": "List of titles used to answer the question"
        },
        "source": {
            "type": "array",
            "items": {"type": "string"},
            "description": "List of sources used to answer the question"
        },
        "segment": {
            "type": "array",
            "items": {"type": "string"},
            "description": "List of direct verbatim segments from documents that answer the question"
        }
    },
    "required": ["id", "title", "source", "segment"]
}

highlight_prompt = PromptTemplate(
    input_variables=["documents", "question", "generation"],
    template=(
        "You are an advanced assistant for document search and retrieval. You are provided with:\n"
        "1. A question.\n"
        "2. A generated answer based on the question.\n"
        "3. A set of documents that were referenced in generating the answer.\n\n"
        "Your task is to identify and extract the exact inline segments from the provided documents "
        "that directly correspond to the content used to generate the given answer. "
        "The extracted segments must be verbatim snippets from the documents, "
        "ensuring a word-for-word match with the text in the provided documents.\n\n"
        "Ensure that:\n"
        "- Each segment is an exact match to a part of the document and is fully contained within the document text.\n"
        "- The relevance of each segment to the generated answer is clear.\n"
        "- If you didn't use a specific document, don't mention it.\n\n"
        "Documents: <docs>{documents}</docs>\n"
        "Question: <question>{question}</question>\n"
        "Answer: <answer>{generation}</answer>"
    )
)

highlight_chain = highlight_prompt | llm.with_structured_output(highlight_schema)

lookup = highlight_chain.invoke({
    "documents": formatted_docs,
    "question": question,
    "generation": generation
})

print("Source segments used to generate the answer:\n")
for doc_id, title, source, segment in zip(
    lookup["id"], lookup["title"], lookup["source"], lookup["segment"]
):
    print(f"ID: {doc_id}")
    print(f"Title: {title}")
    print(f"Source: {source}")
    print(f"Segment: {segment}")
    print("-" * 80)

Source segments used to generate the answer:

ID: Agentic Design Patterns Part 5, Multi-Agent Collaboration Prompting an LLM to play different roles for different parts of a complex task summons a team of AI agents that can do the job more effectively.
Title: Agentic Design Patterns Part 5, Multi-Agent Collaboration
Source: https://www.deeplearning.ai/the-batch/agentic-design-patterns-part-5-multi-agent-collaboration/?ref=dl-staging-website.ghost.io
Segment: Agentic Design Patterns Part 5, Multi-Agent Collaboration✨ New course! Enroll in A2A: The Agent2Agent ProtocolExplore CoursesAI NewsletterThe BatchAndrew's LetterData PointsML ResearchBlog✨ AI Dev x SF 26CommunityForumEventsAmbassadorsAmbassador SpotlightResourcesMembershipStart LearningWeekly IssuesAndrew's LettersData PointsML ResearchBusinessScienceCultureHardwareAI CareersAboutSubscribeThe BatchLettersArticleAgentic Design Patterns Part 5, Multi-Agent Collaboration Prompting an LLM to play different roles for different parts of

---
## Summary

| Step | What happened |
|---|---|
| 1 | Set up local LLM + embeddings |
| 2 | Loaded 5 web articles, chunked, stored in ChromaDB |
| 3 | Retrieved top-4 similar chunks for the question |
| 4 | **Graded relevance** — filtered out irrelevant chunks |
| 5 | Generated answer from relevant chunks only |
| 6 | **Hallucination check** — verified answer is grounded in documents |
| 7 | **Source highlighting** — identified exact text segments used |

**Key insight:** Reliable RAG doesn't just retrieve-and-generate. It adds a **relevance filter** before generation, a **hallucination check** after generation, and **source tracing** for transparency. These three guardrails make the pipeline much more trustworthy.